# ENVIRONMENT

In [ ]:

! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_API_BASE'] = 'https://openrouter.ai/api/v1'
os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'

# BASICS

In [ ]:
import bs4

from langchainhub import Client

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader

from langchain_chroma import Chroma

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

#### LOAD DOCUMENT 

In [ ]:
loader = WebBaseLoader(
    web_paths = ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs = dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content" , "post-title" , "post-header")
        )
    ),
)

docs = loader.load()


Load the document or website and remove unnecessary stuff like the footer, ads, sidebar, etc.

#### SPLIT

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

split = text_splitter.split_documents(docs)

This particular step breaks the documents into chunks (small parts).

#### EMBEDDING

In [ ]:
vectorstore = Chroma.from_documents(
    documents=split,
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

The chunks are then embedded; they form a vector, and that vector is stored in Chroma.

#### PROMPT

In [ ]:
from langchain_core.load import loads
hub = Client()
prompt = hub.pull("rlm/rag-prompt")
if isinstance(prompt, str):
    prompt = loads(prompt)

In this step, we use a ready-made RAG prompt.

#### LLM

In [ ]:
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0
)

In this step, we call the OpenAI model and send the request to it. A temperature of zero means no creativity, which gives the most factual and consistent answer every time. This is good for RAG because you want accuracy, not randomness.

#### POST PROCESSING

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In this step, we format the document. It is originally in the form of a list, but we format it so that it is easy for the AI model to read as a prompt.

#### CHAIN

In [ ]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

This forms a complete system where the user asks a question and gets the response.

#### QUESTION

In [ ]:
response = rag_chain.invoke(
    "What is Task Decomposition?"
)

print(response)


This gives us the final response to the question.

### How the RAG Pipeline Works

```
┌────────────────────────────────────────────────────────────────────────┐
│                         RAG PIPELINE FLOWCHART                         │
└────────────────────────────────────────────────────────────────────────┘

                        INDEXING PHASE

  ┌──────────────────┐      Raw Docs      ┌──────────────────────┐
  │  Load            │ ─────────────────► │  Split               │
  │ (WebBaseLoader)  │                    │ (RecursiveCharacter  │
  └──────────────────┘                    │  TextSplitter)       │
                                          └──────────┬───────────┘
                                                     │
                                                     │  Text Chunks
                                                     ▼
                                          ┌─────────────────────┐
                                          │  Embed              │
                                          │ (OpenAIEmbeddings)  │
                                          └──────────┬──────────┘
                                                     │
                                                     │  Vectors
                                                     ▼
                                          ┌───────────────────────┐
                                          │  Store                │
                                          │ (Chroma VectorStore)  │
                                          └───────────┬───────────┘
                                                      │
                  RETRIEVAL & GENERATION PHASE        │
                                                      │
     ┌──────────────────┐                             │
     │  User Query      │                             │
     └────────┬─────────┘                             │
              │                                       │
              ▼                                       │
  ┌───────────────────────┐                           │
  │  Retrieve             │◄──────────────────────────┘
  │ (Similarity Search)   │   Fetch Context
  └───────────┬───────────┘
              │
              │  Format Docs + Prompt + Context
              ▼
  ┌───────────────────────┐
  │  Generate             │
  │ (LLM / GPT-3.5-Turbo) │
  └───────────┬───────────┘
              │
              ▼
  ┌───────────────────────┐
  │  Output Answer        │
  └───────────────────────┘
```

**Key Insight:**
- **Indexing Phase** loads, splits, embeds, and stores document chunks in Chroma.
- **Retrieval Phase** searches Chroma for relevant chunks using similarity search.
- **Generation Phase** combines the retrieved context with the prompt and sends it to the LLM for a final answer.

# INDEXING

#### DOCUMENTS


In [ ]:
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

#### COUNT TOKEN

In [ ]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

num_tokens_from_string(question, "cl100k_base")

#### TEXT EMBEDDING

In [ ]:
from langchain_openai import OpenAIEmbeddings
embd = OpenAIEmbeddings()
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
len(query_result)

#### COSINE SIMILARITY

In [ ]:
import numpy as np
def cosine_similarity(vec1,vec2):
    dot_product = np.dot(vec1,vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product/(norm_vec1*norm_vec2)

similarity = cosine_similarity(query_result,document_result)
print(similarity)

#### LOAD DOCUMENT

In [ ]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths= ("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs= dict(
        parse_only = bs4.SoupStrainer(
            class_ = ("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

#### SPLITTER

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size = 300,
    chunk_overlap = 50
)
splits = text_splitter.split_documents(blog_docs)

#### VECTORSTORES

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=splits,embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever()

# RETRIEVAL

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(documents=splits , embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k":1})

docs = retriever.invoke("What is task documentation?")

len(docs)

# GENERATION

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
 
template = """Answer the question based only on the following context:
{context}

Question:{question}
"""

prompt = ChatPromptTemplate.from_template(template)
prompt

In [ ]:
llm = ChatOpenAI(model="gpt-3.5-turbo",temperature=0)

In [ ]:
chain = prompt|llm

In [ ]:
chain.invoke({"context":docs,"question":"What is task decomposition"})

In [ ]:
from langchain_core.load import loads
hub = Client()
prompt_hub_rag = hub.pull("rlm/rag-prompt")

In [ ]:
prompt_hub_rag

#### RAG CHAIN

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")